# 04 — Q1: previsão do preço do mês seguinte

**Questão:** é possível prever o preço do mês seguinte de cada produto melhor do que simplesmente repetir o preço atual?

**Método (decisões 3, 6 e 7):**
- série: preço geral (mediana do PPK entre mercados) por produto e mês;
- entradas: preços dos meses anteriores (lags); o custo do produtor (CONAB) foi testado no notebook 03 e ficou fora dos modelos finais (decisão 5);
- modelos: baseline ingênuo, regressão linear, KNN, ensemble (média ponderada) e Random Forest (modelo extra, decisão 6); Gradient Boosting entra só como comparação informal, não é modelo oficial;
- avaliação **sempre com divisão temporal** (treino nos meses antigos, teste nos recentes, ou walk-forward). **Proibido k-fold embaralhado.**
- hiperparâmetros (número de lags, k do KNN) escolhidos numa validação separada dentro do treino, nunca no teste;
- métricas em valor (MAE, MSE) e de direção sobe/não sobe (matriz de confusão, acurácia, precisão, revocação), reportadas separadamente.

Conclusão negativa (nenhum modelo bate o baseline) também é resultado e deve ser reportada com justificativa.

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote src/ a partir da pasta notebooks/
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config
import src.data_loading
import src.aggregation
import src.features
import src.evaluation
import src.q1_models

In [2]:
base_limpa = src.data_loading.carregar_base_limpa()
geral = src.aggregation.serie_geral(base_limpa)
geral.head()

,produto,mes,preco,n_mercados
0,Arroz,2022-09,4.948,29
1,Arroz,2022-10,4.969,28
2,Arroz,2022-11,4.893,27
3,Arroz,2022-12,5.044,28
4,Arroz,2023-01,5.058,28


### Separar os meses de treino antes de escolher hiperparâmetros

`escolher_hiperparametros` só pode ver os meses de treino (decisão 7: nunca
olhar o teste). `avaliar_modelo`, mais abaixo, recebe a base com **todos** os
meses, mas corta por `primeiro_mes_teste` (`evaluation.divisao_por_data`, não
mais `walk_forward`): assim todo produto é testado exatamente nos mesmos
meses (37 a 48), mesmo quando um `n_lags` maior consome mais linhas iniciais
da série do produto (ver decisão 7, atualização, e `docs/pendencias_parte_c.md`
item 1).

In [3]:
meses_ordenados = sorted(geral["mes"].unique())
meses_treino = meses_ordenados[:config.MESES_TREINO]
geral_treino = geral[geral["mes"].isin(meses_treino)]
primeiro_mes_teste = meses_ordenados[config.MESES_TREINO]  # mesmo corte de conab_analise.comparar_modelos_com_sem_conab

len(meses_ordenados), len(meses_treino)

(48, 36)

In [4]:
linhas_resultado = []
linhas_extrapolacao = []

for produto in config.PRODUTOS:
    hiperparametros = src.q1_models.escolher_hiperparametros(geral_treino, produto)
    n_lags = hiperparametros["n_lags"]
    k = hiperparametros["k"]
    pesos = src.q1_models.escolher_pesos_ensemble(geral_treino, produto, n_lags, k)

    base = src.features.montar_base_q1(geral, produto, n_lags)
    colunas_entrada = [f"lag_{i}" for i in range(1, n_lags + 1)]

    linhas_extrapolacao.append({"produto": produto, **src.q1_models.diagnostico_extrapolacao(base)})

    modelos = {
        "baseline": src.q1_models.BaselineIngenuo(),
        "regressao_linear": src.q1_models.criar_regressao_linear(),
        "knn": src.q1_models.criar_knn(k),
    }
    modelos["ensemble"] = src.q1_models.criar_ensemble(modelos, pesos)
    modelos["random_forest"] = src.q1_models.criar_random_forest()
    modelos["gradient_boosting"] = src.q1_models.criar_gradient_boosting()

    for nome_modelo, modelo in modelos.items():
        metricas = src.evaluation.avaliar_modelo(modelo, base, colunas_entrada, "alvo", primeiro_mes_teste)
        linha = metricas.iloc[0].to_dict()
        linha.update({"produto": produto, "modelo": nome_modelo, "n_lags": n_lags, "k": k})
        linhas_resultado.append(linha)

tabela_resultados = pd.DataFrame(linhas_resultado)[
    ["produto", "modelo", "n_lags", "k", "MAE", "MSE", "acuracia", "precisao", "revocacao", "matriz_confusao"]
]
tabela_resultados.sort_values(["produto", "modelo"]).reset_index(drop=True)

,produto,modelo,n_lags,k,MAE,MSE,acuracia,precisao,revocacao,matriz_confusao
0,Arroz,baseline,6,1,0.281364,0.133482,0.454545,0.00,0.000000,"[[5, 0], [6, 0]]"
1,Arroz,ensemble,6,1,0.311997,0.157506,0.454545,0.50,0.666667,"[[1, 4], [2, 4]]"
2,Arroz,gradient_boosting,6,1,0.330984,0.167347,0.636364,0.60,1.000000,"[[1, 4], [0, 6]]"
3,Arroz,knn,6,1,0.474091,0.284412,0.454545,0.50,0.666667,"[[1, 4], [2, 4]]"
4,Arroz,random_forest,6,1,0.296843,0.131142,0.636364,0.60,1.000000,"[[1, 4], [0, 6]]"
...,...,...,...,...,...,...,...,...,...,...
79,Óleo,ensemble,3,9,0.213793,0.078850,0.545455,0.00,0.000000,"[[6, 0], [5, 0]]"
80,Óleo,gradient_boosting,3,9,0.206891,0.078890,0.727273,0.75,0.600000,"[[5, 1], [2, 3]]"
81,Óleo,knn,3,9,0.252839,0.120344,0.545455,0.50,0.200000,"[[5, 1], [4, 1]]"
82,Óleo,random_forest,3,9,0.145145,0.040601,0.636364,0.60,0.600000,"[[4, 2], [2, 3]]"


### Random Forest e Gradient Boosting não extrapolam

Decisão 6: árvores preveem pela média de valores vistos no treino, então não
conseguem prever acima do maior preço já visto. Em produtos com tendência de
alta, isso faz Random Forest e Gradient Boosting subestimarem o teste por um
motivo estrutural, não por falta de padrão nos dados. A tabela abaixo mostra,
por produto, se o preço máximo do teste passou do máximo do treino.

In [5]:
pd.DataFrame(linhas_extrapolacao)

,produto,max_alvo_treino,max_alvo_teste,extrapola
0,Café,73.980000,72.680000,False
1,Arroz,6.958000,5.127500,False
2,Feijão,9.935000,10.320000,True
3,Leite,5.790000,5.990000,True
4,Manteiga,77.700000,72.600000,False
5,Farinha de Trigo,6.540000,5.490000,False
6,Óleo,9.730556,9.655575,False
7,Açúcar,4.840000,4.915000,True
8,Tomate,13.945000,13.395000,False
9,Batata,11.990000,9.990000,False
